# Exploracao dos dados de Picos

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if (PROJECT_DIR / "src" / "scrap_indicadores").exists():
    SRC_DIR = PROJECT_DIR / "src"
elif (PROJECT_DIR / "scrap_indicadores" / "src" / "scrap_indicadores").exists():
    SRC_DIR = PROJECT_DIR / "scrap_indicadores" / "src"
else:
    SRC_DIR = PROJECT_DIR.parent / "src"
sys.path.insert(0, str(SRC_DIR))

import pandas as pd
from IPython.display import display
from pysus.api.client import PySUS

from scrap_indicadores.extract_picos import fetch_dataset, fetch_sinan_dengue_picos
from scrap_indicadores.picos_indicators import (
    PICOS_CODE,
    calculate_picos_indicators,
    filter_picos,
    summarize_sinan_dengue_indicators,
)

In [ ]:
STATE = "PI"
YEAR = 2024
YEARS_TO_CHECK = range(2020, 2026)

DATASETS = [
    {"label": "SIM - Mortalidade", "dataset": "sim", "state": STATE, "group": None},
    {"label": "SIH - Internacoes", "dataset": "sih", "state": STATE, "group": None},
    {"label": "SINAN - Dengue", "dataset": "sinan", "state": None, "group": "DENG", "file_prefix": "DENG"},
    {"label": "PNI - Imunizacao", "dataset": "pni", "state": STATE, "group": None},
]

print(f"Municipio: Picos-PI ({PICOS_CODE})")
print(f"Ano selecionado para coleta: {YEAR}")

Municipio: Picos-PI (220800)
Ano selecionado para coleta: 2024


In [3]:
async def list_available_files(pysus, dataset_config, years):
    rows = []
    for year in years:
        try:
            files = await pysus.query(
                dataset=dataset_config["dataset"],
                state=dataset_config.get("state"),
                year=year,
                group=dataset_config["group"],
            )
            file_prefix = dataset_config.get("file_prefix")
            if file_prefix:
                files = [file for file in files if file.path.name.startswith(file_prefix)]
        except Exception as exc:
            rows.append({
                "base": dataset_config["label"],
                "ano": year,
                "arquivos": 0,
                "nomes": "",
                "status": f"erro: {exc}",
            })
            continue

        rows.append({
            "base": dataset_config["label"],
            "ano": year,
            "arquivos": len(files),
            "nomes": ", ".join(file.path.name for file in files[:5]),
            "status": "ok" if files else "sem arquivos",
        })
    return rows

async with PySUS() as pysus:
    availability_rows = []
    for dataset_config in DATASETS:
        availability_rows.extend(await list_available_files(pysus, dataset_config, YEARS_TO_CHECK))

availability = pd.DataFrame(availability_rows)
display(availability)

,base,ano,arquivos,nomes,status
0,SIM - Mortalidade,2020,1,DOPI2020.parquet,ok
1,SIM - Mortalidade,2021,1,DOPI2021.parquet,ok
2,SIM - Mortalidade,2022,1,DOPI2022.parquet,ok
3,SIM - Mortalidade,2023,1,DOPI2023.parquet,ok
4,SIM - Mortalidade,2024,1,DOPI2024.parquet,ok
5,SIM - Mortalidade,2025,0,,sem arquivos
6,SIH - Internacoes,2020,12,"RDPI2003.parquet, SPPI2005.parquet, SPPI2004.p...",ok
7,SIH - Internacoes,2021,12,"SPPI2105.parquet, SPPI2109.parquet, SPPI2108.p...",ok
8,SIH - Internacoes,2022,12,"SPPI2207.parquet, RDPI2203.parquet, RDPI2209.p...",ok
9,SIH - Internacoes,2023,12,"SPPI2302.parquet, RDPI2304.parquet, RDPI2307.p...",ok


## Coleta dos dados do ano selecionado

A coleta abaixo baixa os dados do Piaui para o ano definido em `YEAR`. Depois os DataFrames sao filtrados para Picos.

In [4]:
async with PySUS() as pysus:
    df_sim = await fetch_dataset(pysus, "sim", state=STATE, year=YEAR)
    df_sih = await fetch_dataset(
        pysus,
        "sih",
        state=STATE,
        year=YEAR,
        filter_fn=lambda file: file.path.name.startswith("RD"),
    )
    df_sinan = await fetch_sinan_dengue_picos(pysus, year=YEAR)
    df_pni = await fetch_dataset(pysus, "pni", state=STATE, year=YEAR)

raw_sizes = pd.DataFrame([
    {"base": "SIM", "linhas_baixadas": len(df_sim), "colunas": len(df_sim.columns)},
    {"base": "SIH", "linhas_baixadas": len(df_sih), "colunas": len(df_sih.columns)},
    {"base": "SINAN", "linhas_baixadas": len(df_sinan), "colunas": len(df_sinan.columns)},
    {"base": "PNI", "linhas_baixadas": len(df_pni), "colunas": len(df_pni.columns)},
])
display(raw_sizes)

Buscando sim (Grupo: None, Estado: PI, Ano: 2024)...
Buscando sih (Grupo: None, Estado: PI, Ano: 2024)...
Erro ao baixar arquivo public/data/ftp/sinan/DENGBR24.parquet: Unexpected error downloading DENGBR24.parquet: 
Nenhum arquivo nacional de Dengue encontrado para SINAN 2024
Buscando pni (Grupo: None, Estado: PI, Ano: 2024)...
Nenhum arquivo encontrado/baixado para pni


,base,linhas_baixadas,colunas
0,SIM,23476,87
1,SIH,109540,113
2,SINAN,0,0
3,PNI,0,0


## Registros encontrados para Picos

In [5]:
df_picos_sim, sim_municipio_col = filter_picos(df_sim, ["CODMUNRES"])
df_picos_sih, sih_municipio_col = filter_picos(df_sih, ["MUNIC_RES"])
df_picos_sinan, sinan_municipio_col = filter_picos(df_sinan, ["ID_MN_RESI", "CO_MUN_RES"])
df_picos_pni, pni_municipio_col = filter_picos(df_pni, ["CODMUNRES", "CO_MUNICIP"])

picos_sizes = pd.DataFrame([
    {"base": "SIM", "coluna_municipio": sim_municipio_col, "linhas_picos": len(df_picos_sim)},
    {"base": "SIH", "coluna_municipio": sih_municipio_col, "linhas_picos": len(df_picos_sih)},
    {"base": "SINAN", "coluna_municipio": sinan_municipio_col, "linhas_picos": len(df_picos_sinan)},
    {"base": "PNI", "coluna_municipio": pni_municipio_col, "linhas_picos": len(df_picos_pni)},
])
display(picos_sizes)

,base,coluna_municipio,linhas_picos
0,SIM,CODMUNRES,621
1,SIH,MUNIC_RES,1849
2,SINAN,None,0
3,PNI,None,0


## Indicadores resumidos

In [6]:
indicadores = calculate_picos_indicators(df_sim, df_sih, df_sinan, df_pni)

resumo = pd.DataFrame([
    {"base": base.upper(), "indicador": indicador, "valor": valor}
    for base, valores in indicadores.items()
    for indicador, valor in valores.items()
    if indicador != "coluna_municipio"
])
display(resumo)

,base,indicador,valor
0,SIM,obitos_total,621
1,SIM,obitos_mulheres_idade_fertil,36
2,SIM,obitos_maternos,1
3,SIM,obitos_infantis,14
4,SIH,internacoes_total,1849
5,SIH,internacoes_infantis,195
6,SIH,principais_causas_internacao_infantil,"{'J189': 31, 'P599': 30, 'P220': 10, 'M625': 9..."
7,SINAN,casos_dengue_notificados,0
8,SINAN,hospitalizacoes,0
9,SINAN,obitos,0


## Resumos automaticos de indicadores faltantes

Use esta tabela para descobrir indicadores que ainda nao viraram funcao formal. Ela mostra distribuicoes de colunas importantes quando elas existem na base.

In [7]:
sinan_resumos = summarize_sinan_dengue_indicators(df_picos_sinan)
sinan_resumos_df = pd.DataFrame([
    {"coluna": coluna, "valor": valor, "quantidade": quantidade}
    for coluna, valores in sinan_resumos.items()
    for valor, quantidade in valores.items()
])
display(sinan_resumos_df)

""


## Colunas disponiveis por base

In [8]:
columns_by_dataset = pd.DataFrame([
    {"base": "SIM", "coluna": column} for column in df_sim.columns
] + [
    {"base": "SIH", "coluna": column} for column in df_sih.columns
] + [
    {"base": "SINAN", "coluna": column} for column in df_sinan.columns
] + [
    {"base": "PNI", "coluna": column} for column in df_pni.columns
])
display(columns_by_dataset)

,base,coluna
0,SIM,ORIGEM
1,SIM,TIPOBITO
2,SIM,DTOBITO
3,SIM,HORAOBITO
4,SIM,NATURAL
...,...,...
195,SIH,TPDISEC5
196,SIH,TPDISEC6
197,SIH,TPDISEC7
198,SIH,TPDISEC8


## Amostras dos dados de Picos

In [9]:
def show_sample(title, df, preferred_columns):
    print(title)
    if df.empty:
        print("Nenhum registro encontrado para Picos.")
        return
    columns = [column for column in preferred_columns if column in df.columns]
    if not columns:
        columns = list(df.columns[:12])
    display(df[columns].head(10))

show_sample("SIM - Mortalidade", df_picos_sim, ["CODMUNRES", "IDADE", "SEXO", "CAUSABAS", "OBITOMAT", "DTOBITO"])
show_sample("SIH - Internacoes", df_picos_sih, ["MUNIC_RES", "IDADE", "DIAG_PRINC", "DT_INTER", "DT_SAIDA"])
show_sample("SINAN - Dengue", df_picos_sinan, ["ID_MN_RESI", "CO_MUN_RES", "DT_NOTIFIC", "DT_SIN_PRI", "CLASSI_FIN"])
show_sample("PNI - Imunizacao", df_picos_pni, ["CODMUNRES", "CO_MUNICIP", "DT_VACINA", "VACINA", "IMUNO"])


SIM - Mortalidade


,CODMUNRES,IDADE,SEXO,CAUSABAS,DTOBITO
3,220800,481,1,B573,01012024
20,220800,301,2,Q210,01012024
56,220800,463,1,E149,01012024
116,220800,464,1,K566,02012024
150,220800,476,2,C56,03012024
205,220800,485,2,I219,03012024
237,220800,436,1,X934,04012024
298,220800,477,2,J189,05012024
302,220800,462,1,N119,05012024
310,220800,435,2,C538,05012024


SIH - Internacoes


,MUNIC_RES,IDADE,DIAG_PRINC,DT_INTER,DT_SAIDA
1315,220800,29,K439,20240707,20240709
2482,220800,31,K810,20240701,20240704
2485,220800,62,D391,20240707,20240709
8672,220800,25,O759,20240522,20240524
8682,220800,92,D689,20240503,20240506
8692,220800,12,N390,20240607,20240609
8704,220800,5,J459,20240724,20240726
8705,220800,44,S667,20240530,20240603
8706,220800,63,K460,20240529,20240530
8708,220800,1,J189,20240502,20240505


SINAN - Dengue
Nenhum registro encontrado para Picos.
PNI - Imunizacao
Nenhum registro encontrado para Picos.
